In [1]:
!pip install onnxruntime -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.9 MB/s eta 0:00:00


In [2]:
!pip install omegaconf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# No system packages required beyond pip installs above

In [ ]:
import re
import unicodedata
import string
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from typing import List
import nltk
nltk.download('punkt_tab', quiet=True)

In [5]:
from typing import List

import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
from sentencepiece import SentencePieceProcessor

In [6]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
# Kaggle: attach 'sentences' dataset (output of data_prep_new.ipynb).
DATA_PATH = '/kaggle/input/sentences/sentences_full.csv'

In [ ]:
df = pd.read_csv(DATA_PATH)

# Handle legacy sentences.csv columns
if 'status' in df.columns:
    df = df[df['status'] != 'needs correction']
if 'corrected' in df.columns:
    df = df[~df['corrected'].isna()]
for col in ['prev_sent', 'next_sent']:
    if col not in df.columns:
        df[col] = ''
df = df.dropna(subset=['text', 'corrected'])
print(f'Loaded {len(df)} rows')

In [ ]:
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    return re.sub(r'\s+', ' ', text).strip()

df['text']      = df['text'].progress_apply(normalize)
df['corrected'] = df['corrected'].progress_apply(normalize)
df['prev_sent'] = df['prev_sent'].fillna('').apply(normalize)
df['next_sent'] = df['next_sent'].fillna('').apply(normalize)

In [ ]:
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
from sentencepiece import SentencePieceProcessor
import onnxruntime as ort

# 1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase
REPO = '1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase'
spe_path    = hf_hub_download(repo_id=REPO, filename='sp.model')
onnx_path   = hf_hub_download(repo_id=REPO, filename='model.onnx')
config_path = hf_hub_download(repo_id=REPO, filename='config.yaml')

sp_model: SentencePieceProcessor = SentencePieceProcessor(spe_path)
ort_sess: ort.InferenceSession   = ort.InferenceSession(onnx_path)
cfg         = OmegaConf.load(config_path)
pre_labels: List[str]  = cfg.pre_labels
post_labels: List[str] = cfg.post_labels
null_tok    = cfg.get('null_token',   '<NULL>')
acronym_tok = cfg.get('acronym_token','<ACRONYM>')

In [ ]:
def add_punct(input_text: str) -> str:
    input_ids = [sp_model.bos_id()] + sp_model.EncodeAsIds(input_text) + [sp_model.eos_id()]
    ids_arr   = np.array([input_ids])
    pre_p, post_p, cap_p, sbd_p = ort_sess.run(None, {'input_ids': ids_arr})
    pre_p, post_p, cap_p, sbd_p = pre_p[0].tolist(), post_p[0].tolist(), cap_p[0].tolist(), sbd_p[0].tolist()
    output_texts, cur_chars = [], []
    for tok_idx in range(1, len(input_ids) - 1):
        token      = sp_model.IdToPiece(input_ids[tok_idx])
        pre_label  = pre_labels[pre_p[tok_idx]]
        post_label = post_labels[post_p[tok_idx]]
        if token.startswith('\u2581') and cur_chars:
            cur_chars.append(' ')
        if pre_label != null_tok:
            cur_chars.append(pre_label)
        char_start = 1 if token.startswith('\u2581') else 0
        for ci, ch in enumerate(token[char_start:], start=char_start):
            cur_chars.append(ch.upper() if cap_p[tok_idx][ci] else ch)
            if post_label == acronym_tok:
                cur_chars.append('.')
        if post_label not in (null_tok, acronym_tok):
            cur_chars.append(post_label)
        if sbd_p[tok_idx]:
            output_texts.append(''.join(cur_chars))
            cur_chars.clear()
    if cur_chars:
        output_texts.append(''.join(cur_chars))
    return ' '.join(output_texts)


def restore_punct(text: str) -> str:
    cleaned = ' '.join(w for w in word_tokenize(str(text)) if w not in string.punctuation)
    if not cleaned.strip():
        return text
    restored = add_punct(cleaned)
    return re.sub(r'\s+', ' ', restored).strip()

In [ ]:
df['text']      = df['text'].progress_apply(restore_punct)
df['corrected'] = df['corrected'].progress_apply(restore_punct)
print('Punctuation restored')
df[['text', 'corrected']].head()

In [ ]:
cyrillic = re.compile(r'[а-яА-ЯёЁ]')
df = df[df['text'].str.contains(cyrillic, na=False)]
df = df[df['corrected'].str.contains(cyrillic, na=False)]
df = df[df['text'] != df['corrected']]
df = df.drop_duplicates(subset=['text', 'corrected'])
df = df.reset_index(drop=True)
print(f'Pairs after filtering: {len(df)}')

In [ ]:
out_cols = ['text', 'corrected', 'prev_sent', 'next_sent']
df[out_cols].to_csv('sentences_preprocessed.csv', index=False)
print(f'Saved sentences_preprocessed.csv: {len(df)} rows')
df[['text', 'corrected']].head()